In [1]:
import os
os.chdir("/kaggle/working")
!pip uninstall -y torchao
!pip install -q deepspeed peft accelerate sentencepiece opensmile
if not os.path.exists("SpeechCueLLM"):
    !git clone -q https://github.com/zehuiwu/SpeechCueLLM.git

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 16.0 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
token = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download("meta-llama/Llama-3.2-3B-Instruct", local_dir="/kaggle/temp/llama32-3b",
                  token=token, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/kaggle/temp/llama32-3b'

In [3]:
import json, re
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

def sub(path, a, b):
    s = open(path).read()
    if a in s and b not in s:
        open(path, "w").write(s.replace(a, b))

M = "main.py"
sub(M, "model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
       "model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, torch_dtype=torch.float16, low_cpu_mem_usage=True)")
sub(M, '        deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
       '        deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
sub(M, "    model.gradient_checkpointing_enable()",
       "    model.gradient_checkpointing_enable()\n    model.enable_input_require_grads()")
sub(M, "    optimizer_grouped_parameters = getOptimizerGroup(model=model)\n",
       "    optimizer_grouped_parameters = getOptimizerGroup(model=model)\n"
       "    optimizer_grouped_parameters = [g for g in optimizer_grouped_parameters if len(g['params']) > 0]\n"
       "    print('optimizer groups:', [len(g['params']) for g in optimizer_grouped_parameters])\n")
sub(M, "    model_engine, optimizer, train_dataloader, lr_scheduler = deepspeed.initialize(",
       '    for _k in ("mics_shard_size", "mics_hierarchical_params_gather"):\n'
       '        deepspeed_config["zero_optimization"].pop(_k, None)\n'
       "    model_engine, optimizer, train_dataloader, lr_scheduler = deepspeed.initialize(")

p = "data_utils/deepspeed_config.json"
c = json.load(open(p))
c["bfloat16"]["enabled"] = False
c["fp16"]["enabled"] = True
c["zero_optimization"]["stage"] = 2
json.dump(c, open(p, "w"), indent=2)

EPOCHS, DATA_PCT, SEED = 5, 1.0, 11
sh = "train_and_inference.sh"
s = open(sh).read()
def setvar(name, val):
    global s
    s = re.sub(rf"^{name}=.*$", f"{name}={val}", s, count=1, flags=re.M)
setvar("MODEL_NAME", "'LLaMA3-instruct'"); setvar("Experiments_setting", "'lora'")
setvar("num_train_epochs", EPOCHS); setvar("data_percent", DATA_PCT)
setvar("SEED", SEED); setvar("graphics_card", 2)
setvar("task", "'B0_text_only'")
s = s.replace("MODEL_PATH='LLaMA3-instruct MODELPATH'", "MODEL_PATH='/kaggle/temp/llama32-3b'")
if "--gradient_checkpointing" not in s:
    s = s.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open(sh, "w").write(s)

!python -m py_compile main.py && echo "syntax ok"
!grep -n "MODEL_PATH='/kaggle" train_and_inference.sh

syntax ok
143:        MODEL_PATH='/kaggle/temp/llama32-3b'


In [4]:
!timeout 11h bash train_and_inference.sh > ../run_log.txt 2>&1
!tr '\r' '\n' < ../run_log.txt | grep -E "best_f1|current_f1|history" | tail -n 10

history: [58.586, 65.752, 67.861, 67.735]
best_f1: 67.491
current_f1: 66.779
history: [59.134, 64.574, 67.491, 66.779]
 80%|████████  | 163/203 [05:10<01:14,  1.85s/it]best_f1: 67.861
current_f1: 66.333
history: [58.586, 65.752, 67.861, 67.735, 66.333]
best_f1: 67.491
current_f1: 67.09
history: [59.134, 64.574, 67.491, 66.779, 67.09]
